In [9]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from scipy import stats
import seaborn as sns
import rasterio
from pyproj import Transformer
import pyogrio
from pathlib import Path

In [4]:
fechas_amatitlan = [
    "2025-01-28",
    "2025-04-15",
    "2025-04-28",
    "2025-11-24",
    "2026-01-08",
    "2026-02-02",
    "2026-02-07",
    "2026-03-29",
    "2026-04-13",
    "2026-04-28",
    "2026-06-19"
]

fechas_atitlan = [
    "2025-01-18",
    "2025-04-13",
    "2025-05-13",
    "2025-07-17",
    "2025-11-21",
    "2025-12-29",
    "2026-02-12",
    "2026-03-24",
    "2026-04-13",
    "2026-04-28",
    "2026-07-22"
]

In [5]:
#La siguiente función devuelve el promedio de la cianobaccteria y es la misma función del inciso 4
def promedio_cianobacteria(ruta_raster):

    with rasterio.open(ruta_raster) as src:
        
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        data[~np.isfinite(data)] = np.nan
        promedio = np.nanmean(data)

    return promedio

#LA siguiente realiza el procesamiento en todas las fechas; es la misma función del inciso 4
def procesar_lago(nombre_lago, carpeta, fechas, nombre):
    resultados = []

    for fecha in fechas:
        ruta = Path(carpeta) / f"{nombre}_{fecha}_indices.tif"

        promedio = promedio_cianobacteria(ruta)

        resultados.append({
            "lago": nombre_lago,
            "fecha": fecha,
            "cyano_promedio": promedio
        })
        
    return pd.DataFrame(resultados)

In [7]:
#La siguiente función calcula el NDVI y la otra el promedio de este
def ndvi(ruta_red, ruta_nir):

    red = leer_raster(ruta_red)
    nir = leer_raster(ruta_nir)

    # Evitar división entre cero
    denominador = nir + red

    ndvi = np.divide(
        nir - red,
        denominador,
        out=np.full_like(nir, np.nan),
        where=denominador != 0
    )

    # Rango teórico válido
    ndvi[(ndvi < -1) | (ndvi > 1)] = np.nan

    return ndvi
def mndvi(ruta_red, ruta_nir):

    ndvi = calcular_ndvi(
        ruta_red,
        ruta_nir
    )

    return np.nanmean(ndvi)

#La siguiente función calcula el NDWI y su correspondiente promedio

def ndwi(ruta_green, ruta_nir):

    green = leer_raster(ruta_green)
    nir = leer_raster(ruta_nir)

    denominador = green + nir

    ndwi = np.divide(
        green - nir,
        denominador,
        out=np.full_like(green, np.nan),
        where=denominador != 0
    )

    ndwi[(ndwi < -1) | (ndwi > 1)] = np.nan

    return ndwi

def mndwi(ruta_green, ruta_nir):

    ndwi = calcular_ndwi(
        ruta_green,
        ruta_nir
    )

    return np.nanmean(ndwi)

In [10]:
#Se procceden a crear dataframes con el índice promedio, es parte del proceso usado en el inciso 4
df_amatitlan = procesar_lago(nombre_lago="Amatitlán", carpeta="data/processed/amatitlan", fechas=fechas_amatitlan, nombre='amatitlan')
df_atitlan = procesar_lago(nombre_lago="Atitlán", carpeta="data/processed/atitlan", fechas=fechas_atitlan, nombre='atitlan')

df = pd.concat([df_amatitlan, df_atitlan], ignore_index=True)
df["fecha"] = pd.to_datetime(df["fecha"])

In [11]:
df_amatitlan

,lago,fecha,cyano_promedio
0,Amatitlán,2025-01-28,4.428946
1,Amatitlán,2025-04-15,4.528544
2,Amatitlán,2025-04-28,4.255442
3,Amatitlán,2025-11-24,5.468830
4,Amatitlán,2026-01-08,6.688481
5,Amatitlán,2026-02-02,4.287532
6,Amatitlán,2026-02-07,4.302523
7,Amatitlán,2026-03-29,6.431256
8,Amatitlán,2026-04-13,6.738171
9,Amatitlán,2026-04-28,10.026723


In [12]:
df_atitlan

,lago,fecha,cyano_promedio
0,Atitlán,2025-01-18,0.386553
1,Atitlán,2025-04-13,1.783338
2,Atitlán,2025-05-13,1.293994
3,Atitlán,2025-07-17,0.820357
4,Atitlán,2025-11-21,0.271335
5,Atitlán,2025-12-29,0.534299
6,Atitlán,2026-02-12,0.916540
7,Atitlán,2026-03-24,1.265022
8,Atitlán,2026-04-13,2.128609
9,Atitlán,2026-04-28,1.855590
